In [ ]:
# ==========================================================
# CELL 1 — SETTINGS (change numbers here, nothing else)
# ==========================================================

HOW_MANY_DIALOGUES = 1       # how many conversations to test (max 20)
HOW_MANY_QUESTIONS_EACH = 5   # how many questions per conversation (max 20)

In [ ]:
# ==========================================================
# CELL 2 — Setup (load model once)
# ==========================================================
from brain import Brain
from agent import Agent
from tracer import Tracer
from tools import ALL_TOOLS
from memory.memory_manager import MemoryManager
import config

brain_gpu0 = Brain(TOKEN, device="cuda:0")
brain_gpu1 = Brain(TOKEN, device="cuda:1")

def make_generate_fn(brain_instance):
    def generate(prompt):
        return brain_instance.think([{"role": "user", "content": prompt}])["text"]
    return generate

local_generate_0 = make_generate_fn(brain_gpu0)
local_generate_1 = make_generate_fn(brain_gpu1)

In [ ]:
# ==========================================================
# CELL 3 — Load the dataset (plain, no groupby)
# ==========================================================
import pandas as pd

all_questions = pd.read_csv("beam_100k_questions.csv")
all_histories = pd.read_csv("beam_100k_histories.csv")

# get the list of conversation IDs, and only keep as many as we want to test
conversation_id_list = all_histories["conversation_id"].unique()
conversation_id_list = conversation_id_list[:HOW_MANY_DIALOGUES]

print("Testing", len(conversation_id_list), "conversations")

In [ ]:
# ==========================================================
# CELL 4 — Judge function (checks if the answer is correct)
# ==========================================================
import ast

def judge_abstention(agent_answer):
    answer_lower = agent_answer.lower()
    for marker in config.ABSTENTION_MARKERS:
        if marker in answer_lower:
            return True
    return False

def judge_with_rubric(agent_answer, rubric_text):
    try:
        rubric_list = ast.literal_eval(rubric_text)
    except Exception:
        rubric_list = [rubric_text]

    answer_lower = agent_answer.lower()
    matches = 0
    for point in rubric_list:
        if str(point).lower()[:20] in answer_lower:
            matches = matches + 1

    needed = max(1, len(rubric_list) // 2)
    return matches >= needed

def judge(question_type, agent_answer, rubric_text):
    if question_type == "abstention":
        return judge_abstention(agent_answer)
    else:
        return judge_with_rubric(agent_answer, rubric_text)

In [ ]:
# ==========================================================
# CELL 5 (FIXED) — Main loop, TRUE parallel across 2 GPUs using PROCESSES
# ==========================================================
import os
import multiprocessing as mp
import pandas as pd

# beam_worker.py must sit in PROJECT_DIR alongside brain.py, agent.py, etc.
import sys
sys.path.insert(0, "/kaggle/working/Harness-Memory-")
from beam_worker import run_some_dialogues

if __name__ == "__main__":
    PROJECT_DIR = "/kaggle/working/Harness-Memory-"
    HISTORIES_CSV = "/kaggle/input/datasets/sadi2oo3q/beam-dataset-100k/beam_100k_histories.csv"
    QUESTIONS_CSV = "/kaggle/input/datasets/sadi2oo3q/beam-dataset-100k/beam_100k_questions.csv"
    HOW_MANY_DIALOGUES = 20
    HOW_MANY_QUESTIONS_EACH = 10

    mp.set_start_method("spawn", force=True)

    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")

    all_histories = pd.read_csv(HISTORIES_CSV)
    conversation_id_list = list(all_histories["conversation_id"].unique())[:HOW_MANY_DIALOGUES]

    half_point = len(conversation_id_list) // 2
    first_half = conversation_id_list[:half_point]
    second_half = conversation_id_list[half_point:]

    print("GPU 0 will do", len(first_half), "conversations")
    print("GPU 1 will do", len(second_half), "conversations")

    p0 = mp.Process(target=run_some_dialogues, args=(
        first_half, "cuda:0", "GPU0", PROJECT_DIR, HISTORIES_CSV, QUESTIONS_CSV,
        HOW_MANY_QUESTIONS_EACH, hf_token, "beam_eval_results_gpu0.csv",
    ))
    p1 = mp.Process(target=run_some_dialogues, args=(
        second_half, "cuda:1", "GPU1", PROJECT_DIR, HISTORIES_CSV, QUESTIONS_CSV,
        HOW_MANY_QUESTIONS_EACH, hf_token, "beam_eval_results_gpu1.csv",
    ))

    p0.start()
    p1.start()
    p0.join()
    p1.join()

    # FIX: fail loudly if a worker produced no results, instead of quietly
    # merging into an empty frame and hitting KeyError later.
    for f in ("beam_eval_results_gpu0.csv", "beam_eval_results_gpu1.csv"):
        if not os.path.exists(f):
            raise RuntimeError(f"{f} was never created — that worker crashed. Check the printed traceback above.")

    print("\nDone. Merging results...")
    df0 = pd.read_csv("beam_eval_results_gpu0.csv")
    df1 = pd.read_csv("beam_eval_results_gpu1.csv")
    merged = pd.concat([df0, df1], ignore_index=True)

    results_file = "beam_eval_results.csv"   # so Cell 6 can find it
    merged.to_csv(results_file, index=False)
    print("Overall accuracy:", round(merged["correct"].mean() * 100, 1), "%")

In [ ]:
# ==========================================================
# CELL 6 — See the score
# ==========================================================
results = pd.read_csv(results_file)

print("Overall accuracy:", round(results["correct"].mean() * 100, 1), "%")
print()
print(results.groupby("q_type")["correct"].mean())